# Bronze Layer — Raw Ingestion from S3

In [0]:
import boto3
import json
import pandas as pd

aws_access_key = dbutils.secrets.get(scope="crypto-pipeline-scope", key="AWS_ACCESS_KEY_ID")
aws_secret_key = dbutils.secrets.get(scope="crypto-pipeline-scope", key="AWS_SECRET_ACCESS_KEY")
s3_bucket      = dbutils.secrets.get(scope="crypto-pipeline-scope", key="S3_BUCKET")

s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name="ap-southeast-1"
)

response = s3.list_objects_v2(Bucket=s3_bucket, Prefix="prices/", MaxKeys=5)
print(f"S3 connected! Files found: {response.get('KeyCount', 0)}")
for obj in response.get("Contents", []):
    print(" -", obj["Key"])

In [0]:
def read_json_files_from_s3(prefix):
    all_records = []
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=s3_bucket, Prefix=prefix)
    for page in pages:
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".json"):
                file_obj = s3.get_object(Bucket=s3_bucket, Key=obj["Key"])
                raw_text = file_obj["Body"].read().decode("utf-8")
                data = json.loads(raw_text)
                if isinstance(data, list):
                    all_records.extend(data)
                else:
                    all_records.append(data)
    return all_records


def to_spark_df(records):
    flat = []
    for r in records:
        if isinstance(r, dict) and "records" in r:
            flat.extend(r["records"])
        elif isinstance(r, dict) and "data" in r:
            data = r["data"]
            if isinstance(data, list):
                flat.extend(data)
            else:
                flat.append(r)
        else:
            flat.append(r)

    cleaned = []
    for row in flat:
        cleaned.append({
            k: (json.dumps(v) if isinstance(v, (dict, list)) else v)
            for k, v in row.items()
        })

    return spark.createDataFrame(pd.DataFrame(cleaned))


prices_records = read_json_files_from_s3("prices/")
print(f"Prices records loaded: {len(prices_records)}")
df_prices_raw = to_spark_df(prices_records)
print(f"Columns: {df_prices_raw.columns}")
display(df_prices_raw)

In [0]:
fear_greed_records = read_json_files_from_s3("fear_greed/")
news_records       = read_json_files_from_s3("news/")

print(f"Fear & Greed records: {len(fear_greed_records)}")
print(f"News records: {len(news_records)}")

df_fear_greed_raw = to_spark_df(fear_greed_records)
df_news_raw       = to_spark_df(news_records)

print(f"Fear & Greed columns: {df_fear_greed_raw.columns}")
display(df_fear_greed_raw)

In [0]:
df_prices_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_prices")
df_fear_greed_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_fear_greed")
df_news_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_news")

print("Bronze tables written: bronze_prices, bronze_fear_greed, bronze_news")

In [0]:
#show tables for sanity check
display(spark.sql("SHOW TABLES"))